# RFM Customer Segmentation — Gold Layer

Analisis segmentasi pelanggan menggunakan **RFM (Recency, Frequency, Monetary)** dan **K-Means Clustering**.  
Hasil disimpan ke Gold Layer (`s3a://gold/rfm_segments`) sebagai output akhir pipeline data.

**Alur kerja:**
1. Load data dari Silver Layer
2. EDA — profil distribusi data
3. Hitung metrik RFM per pelanggan
4. Preprocessing — Log Transform → StandardScaler
5. K Selection — Silhouette Score
6. Train model K-Means final
7. Interpretasi & labeling segmen
8. Simpan ke Gold Layer

In [1]:
import sys
from itertools import chain

sys.path.append('/home/jovyan/work')

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from ingestion.spark_client import get_spark_session

# --- Constants ---
SILVER_PATH    = "s3a://silver/integrated_sales/"
GOLD_PATH      = "s3a://gold/rfm_segments"
REFERENCE_YEAR = 2024   # filter: 2 tahun terakhir agar Recency tidak bias data lama
K_OPTIMAL      = 4      # jumlah segmen pelanggan

spark = get_spark_session(app_name="RFM-Segmentation-Gold")
print("Spark Session ready.")

Spark Session ready.


## 1. Load Silver Layer

In [2]:
df_silver = spark.read.parquet(SILVER_PATH)

df_silver.printSchema()
df_silver.groupBy("year").count().orderBy("year").show()

root
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- invoice_id: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- revenue: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- created_at: date (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email_opt_in: integer (nullable = true)
 |-- sms_opt_in: integer (nullable = true)
 |-- call_opt_in: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- gramm_g: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- year: integer (nullable = true)

+----+-----+
|year|count|
+----+-----+
|2015|87447|
|2016|90483|
|2017|90461|
|2018|90039|
|2019|90242|
|2020|90186|
|2021|90792|
|20

## 2. Exploratory Data Analysis

In [3]:
print("=== Revenue & Quantity Descriptive Stats ===")
df_silver.describe("revenue", "quantity").show()

print("=== Top 10 Highest-Spending Customers (all years) ===")
(
    df_silver
    .groupBy("customer_id")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .show(10)
)

=== Revenue & Quantity Descriptive Stats ===
+-------+------------------+--------------------+
|summary|           revenue|            quantity|
+-------+------------------+--------------------+
|  count|            990923|              990923|
|   mean|39.877505618499114|                 1.0|
| stddev| 69.35344856814966|5.55108159536615E-17|
|    min|               7.0|                   1|
|    max|             569.5|                   1|
+-------+------------------+--------------------+

=== Top 10 Highest-Spending Customers (all years) ===
+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|      26687|       2529.5|
|       1346|       2352.0|
|        503|       2234.5|
|       1012|       2230.5|
|       3276|       2202.5|
|       3111|       2177.0|
|       1067|       2159.5|
|        166|       2124.5|
|        682|       2116.0|
|       1344|       2104.0|
+-----------+-------------+
only showing top 10 rows



## 3. Compute RFM Metrics

Filter hanya `year >= 2024` agar skor Recency tidak bias oleh transaksi lama (data mulai 2015).

| Metrik | Definisi |
|--------|-----------|
| **Recency** | Hari sejak transaksi terakhir (lebih kecil = lebih baik) |
| **Frequency** | Jumlah invoice unik per pelanggan |
| **Monetary** | Total revenue pelanggan |

In [4]:
df_recent = df_silver.filter(F.col("year") >= REFERENCE_YEAR)

max_date = df_recent.select(F.max("invoice_date")).collect()[0][0]
print(f"Reference date (max invoice_date): {max_date}")

rfm = df_recent.groupBy("customer_id").agg(
    F.datediff(F.lit(max_date), F.max("invoice_date")).alias("recency"),
    F.countDistinct("invoice_id").alias("frequency"),
    F.round(F.sum("revenue"), 2).alias("monetary"),
)

print(f"Total customers: {rfm.count():,}")
rfm.orderBy(F.desc("monetary")).show(10)

Reference date (max invoice_date): 2025-12-31
Total customers: 124,108
+-----------+-------+---------+--------+
|customer_id|recency|frequency|monetary|
+-----------+-------+---------+--------+
|     419966|    396|        3|  1036.5|
|     454283|    335|        2|  1026.5|
|     421600|    359|        2|   914.0|
|     424348|    461|        3|   862.5|
|     442666|    421|        2|   844.0|
|     455384|    248|        2|   844.0|
|     392620|    621|        2|   820.0|
|     437672|    334|        2|   786.5|
|     451865|      3|        4|   763.0|
|     364962|     52|        3|   734.5|
+-----------+-------+---------+--------+
only showing top 10 rows



## 4. Preprocessing

Pipeline transformasi sebelum clustering:

1. **Log Transform** — `log(x + 1)` untuk menekan distribusi skewed pada Recency dan Monetary
2. **VectorAssembler** — menggabungkan `r_log, f_log, m_log` menjadi satu vektor fitur
3. **StandardScaler (Z-Score)** — normalisasi ke Mean=0, Std=1 agar K-Means tidak didominasi satu fitur

In [5]:
rfm_log = (
    rfm
    .withColumn("r_log", F.log(F.col("recency") + 1))
    .withColumn("f_log", F.log(F.col("frequency") + 1))
    .withColumn("m_log", F.log(F.col("monetary") + 1))
)

assembler = VectorAssembler(
    inputCols=["r_log", "f_log", "m_log"],
    outputCol="features",
)
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True,
)

preprocessing = Pipeline(stages=[assembler, scaler])
rfm_scaled = preprocessing.fit(rfm_log).transform(rfm_log)

print("=== Log-Transformed Feature Stats ===")
rfm_scaled.describe("r_log", "f_log", "m_log").show()

=== Log-Transformed Feature Stats ===
+-------+------------------+-------------------+------------------+
|summary|             r_log|              f_log|             m_log|
+-------+------------------+-------------------+------------------+
|  count|            124108|             124108|            124108|
|   mean| 5.408786977686396| 0.8556566956194065|3.4812767531685878|
| stddev|1.0644291376727635|0.25716539891758305|0.8577251092441033|
|    min|               0.0| 0.6931471805599453|2.0794415416798357|
|    max| 6.594413459749778|  2.302585092994046| 6.944569252104853|
+-------+------------------+-------------------+------------------+



## 5. K Selection — Silhouette Score

Silhouette Score mengukur seberapa baik setiap titik data cocok dengan cluster-nya sendiri vs cluster lain.  
Rentang: **[-1, 1]** — semakin mendekati 1, semakin baik pemisahan antar cluster.

In [6]:
evaluator = ClusteringEvaluator(featuresCol="scaled_features")

print("K  | Silhouette Score")
print("---|------------------")
scores = {}
for k in range(2, 8):
    model_k = KMeans(featuresCol="scaled_features", k=k, seed=42).fit(rfm_scaled)
    score   = evaluator.evaluate(model_k.transform(rfm_scaled))
    scores[k] = score
    print(f"K={k} | {score:.4f}")

print(f"\nBest mathematical K = {max(scores, key=scores.get)}")
print(f"Selected  K = {K_OPTIMAL}  (4 segmen lebih actionable secara bisnis)")

K  | Silhouette Score
---|------------------
K=2 | 0.5878
K=3 | 0.4466
K=4 | 0.5282
K=5 | 0.5038
K=6 | 0.5423
K=7 | 0.5385

Best mathematical K = 2
Selected  K = 4  (4 segmen lebih actionable secara bisnis)


## 6. Train Final Model — K-Means (K=4)

K=4 dipilih untuk menghasilkan 4 segmen pelanggan yang bermakna secara bisnis,  
meskipun K=2 memiliki Silhouette Score tertinggi secara matematis.

In [7]:
kmeans  = KMeans(featuresCol="scaled_features", k=K_OPTIMAL, seed=42)
model   = kmeans.fit(rfm_scaled)
results = model.transform(rfm_scaled)

print("=== Cluster Centroids (Z-Score space: recency, frequency, monetary) ===")
for i, c in enumerate(model.clusterCenters()):
    print(f"  Cluster {i}: recency={c[0]:.3f}  frequency={c[1]:.3f}  monetary={c[2]:.3f}")

print("\n=== Raw Cluster Summary ===")
(
    results
    .groupBy("prediction")
    .agg(
        F.round(F.avg("recency"),   1).alias("avg_recency"),
        F.round(F.avg("frequency"), 2).alias("avg_frequency"),
        F.round(F.avg("monetary"),  2).alias("avg_monetary"),
        F.count("customer_id").alias("customer_count"),
    )
    .orderBy(F.desc("avg_monetary"))
    .show()
)

=== Cluster Centroids (Z-Score space: recency, frequency, monetary) ===
  Cluster 0: recency=0.413  frequency=-0.616  monetary=-0.997
  Cluster 1: recency=-2.052  frequency=-0.063  monetary=-0.091
  Cluster 2: recency=-0.110  frequency=1.382  monetary=0.791
  Cluster 3: recency=0.425  frequency=-0.632  monetary=0.569

=== Raw Cluster Summary ===
+----------+-----------+-------------+------------+--------------+
|prediction|avg_recency|avg_frequency|avg_monetary|customer_count|
+----------+-----------+-------------+------------+--------------+
|         2|      254.3|         2.42|       77.47|         34745|
|         3|      403.8|          1.0|       65.85|         31500|
|         1|       34.3|         1.38|        39.7|         13586|
|         0|      400.2|         1.01|       13.56|         44277|
+----------+-----------+-------------+------------+--------------+



## 7. Cluster Interpretation & Labeling

Berdasarkan profil RFM setiap cluster:

| Cluster | Avg Recency | Avg Frequency | Avg Monetary | Segment | Rekomendasi Strategi |
|---------|-------------|---------------|--------------|---------|---------------------|
| 2 | ~254 hari | 2.4x | ~77 | **Champions** | Reward & loyalty program |
| 3 | ~404 hari | 1.0x | ~66 | **At Risk** | Win-back campaign |
| 1 | ~34 hari  | 1.4x | ~40 | **Promising** | Nurture & upsell |
| 0 | ~400 hari | 1.0x | ~14 | **Lost** | Re-engagement atau deprioritize |

In [8]:
SEGMENT_MAP = {
    2: "Champions",
    3: "At Risk",
    1: "Promising",
    0: "Lost",
}

segment_expr = F.create_map([F.lit(x) for x in chain(*SEGMENT_MAP.items())])

rfm_final = (
    results
    .select("customer_id", "recency", "frequency", "monetary", "prediction")
    .withColumn("segment", segment_expr[F.col("prediction")])
)

print("=== Final Segment Distribution ===")
(
    rfm_final
    .groupBy("segment")
    .agg(
        F.round(F.avg("recency"),   1).alias("avg_recency"),
        F.round(F.avg("frequency"), 2).alias("avg_frequency"),
        F.round(F.avg("monetary"),  2).alias("avg_monetary"),
        F.count("customer_id").alias("customer_count"),
    )
    .orderBy(F.desc("avg_monetary"))
    .show()
)

rfm_final.show(10)

=== Final Segment Distribution ===
+---------+-----------+-------------+------------+--------------+
|  segment|avg_recency|avg_frequency|avg_monetary|customer_count|
+---------+-----------+-------------+------------+--------------+
|Champions|      254.3|         2.42|       77.47|         34745|
|  At Risk|      403.8|          1.0|       65.85|         31500|
|Promising|       34.3|         1.38|        39.7|         13586|
|     Lost|      400.2|         1.01|       13.56|         44277|
+---------+-----------+-------------+------------+--------------+

+-----------+-------+---------+--------+----------+---------+
|customer_id|recency|frequency|monetary|prediction|  segment|
+-----------+-------+---------+--------+----------+---------+
|     312703|    265|        1|    13.5|         0|     Lost|
|     403389|    227|        3|   163.5|         2|Champions|
|     409026|     14|        2|    30.5|         1|Promising|
|     414256|    164|        1|    55.5|         3|  At Risk|
| 

## 8. Save to Gold Layer

Data disimpan ke MinIO dalam format Parquet, dipartisi berdasarkan `segment`  
agar query downstream bisa memanfaatkan partition pruning.

In [9]:
(
    rfm_final
    .write
    .mode("overwrite")
    .partitionBy("segment")
    .parquet(GOLD_PATH)
)

print(f"Saved to: {GOLD_PATH}")
print("\nRow count per partition:")
rfm_final.groupBy("segment").count().orderBy(F.desc("count")).show()

Saved to: s3a://gold/rfm_segments

Row count per partition:
+---------+-----+
|  segment|count|
+---------+-----+
|     Lost|44277|
|Champions|34745|
|  At Risk|31500|
|Promising|13586|
+---------+-----+



In [10]:
spark.stop()
print("Spark session closed.")

Spark session closed.
